# RequirementClarifierAgent - 多智能体需求澄清与技术方案助手

## 项目简介
使用 HelloAgents 的四个 SimpleAgent 协作，将模糊需求转化为结构化的需求与技术方案报告。

## 作者信息
- GitHub：[@zenith191](https://github.com/zenith191)
- 日期：2026-07-30

## 第1部分：环境配置

In [ ]:
# 首次运行时取消下一行注释安装官方框架
# %pip install -q "hello-agents[all]==0.2.9"

import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, SimpleAgent, ToolRegistry
from hello_agents.tools import Tool, ToolParameter
from IPython.display import Markdown, display

project_name = "zenith191-RequirementClarifierAgent"
candidates = [
    Path.cwd(),
    Path.cwd() / "Co-creation-projects" / project_name,
    Path.cwd().parent / project_name,
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "main.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("未找到项目目录，请从项目目录或仓库根目录启动 Notebook")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.agents import build_agent_team
from src.config import LLMSettings
from src.tools import create_tool_registry
from src.workflow import RequirementClarifierWorkflow

print(f"✅ 环境配置完成：{PROJECT_ROOT}")

## 第2部分：工具定义

项目通过官方 `Tool`、`ToolParameter` 和 `ToolRegistry` 提供两个确定性工具：需求完整度初检和报告结构质检。

In [ ]:
tool_registry = create_tool_registry()
assert isinstance(tool_registry, ToolRegistry)

sample_requirement = (PROJECT_ROOT / "data" / "sample_requirement.txt").read_text(encoding="utf-8")
audit_tool = tool_registry.get_tool("requirement_audit")
audit_result = json.loads(audit_tool.run({"requirement_text": sample_requirement}))

print(audit_result["summary"])
print("澄清问题：")
for question in audit_result["clarifying_questions"]:
    print(f"- {question}")

## 第3部分：智能体构建

四个角色分别负责需求分析、方案设计、风险审查和报告整合。只有 `.env` 中三项 LLM 配置齐全时才连接模型服务。

In [ ]:
required_env = ("LLM_MODEL_ID", "LLM_API_KEY", "LLM_BASE_URL")
api_key = os.getenv("LLM_API_KEY", "").strip()
has_llm_config = (
    all(os.getenv(name, "").strip() for name in required_env)
    and not api_key.casefold().startswith("your_")
)

team = None
workflow = None
if has_llm_config:
    settings = LLMSettings.from_env()
    team = build_agent_team(settings, tool_registry)
    workflow = RequirementClarifierWorkflow(team, tool_registry)
    for role, agent in (
        ("需求分析师", team.analyst),
        ("方案架构师", team.architect),
        ("风险审查员", team.reviewer),
        ("报告整合员", team.synthesizer),
    ):
        print(f"✅ {role}: {type(agent).__name__}")
else:
    print("ℹ️ 未检测到完整 LLM 配置：跳过在线智能体创建，继续离线演示。")

## 第4部分：功能演示

In [ ]:
# 示例1：基础功能——无需 API 密钥的完整度检查
print("=== 示例1：需求完整度初检 ===")
print(f"覆盖率：{audit_result['coverage_percent']}%")
print(f"已覆盖：{'、'.join(audit_result['covered_dimensions'])}")
print(f"待补充：{'、'.join(audit_result['missing_dimensions'])}")

In [ ]:
# 示例2：复杂场景——有密钥时运行四智能体，否则展示仓库示例
print("=== 示例2：多智能体需求澄清 ===")
if workflow is not None:
    result = workflow.run(sample_requirement)
    report_path = workflow.save_report(
        result, PROJECT_ROOT / "outputs" / "requirement_report.md"
    )
    print(f"✅ 在线报告已保存：{report_path}")
    display(Markdown(result.report))
else:
    example_report = (
        PROJECT_ROOT / "outputs" / "requirement_report.md"
    ).read_text(encoding="utf-8")
    print("ℹ️ 当前展示仓库内置示例；填写 .env 后重新运行即可调用真实智能体。")
    display(Markdown(example_report))

## 第5部分：性能评估

In [ ]:
iterations = 200
started_at = time.perf_counter()
for _ in range(iterations):
    audit_tool.run({"requirement_text": sample_requirement})
elapsed_ms = (time.perf_counter() - started_at) * 1000

quality_tool = tool_registry.get_tool("report_quality_check")
example_report = (PROJECT_ROOT / "outputs" / "requirement_report.md").read_text(encoding="utf-8")
quality_result = json.loads(quality_tool.run({"report_text": example_report}))

print(f"确定性初检：{iterations} 次共 {elapsed_ms:.2f} ms，平均 {elapsed_ms / iterations:.3f} ms/次")
print(f"示例需求覆盖率：{audit_result['coverage_percent']}%")
print(f"示例报告结构评分：{quality_result['score']}/100")
print("自动化测试命令：python -m pytest -q")

## 第6部分：总结与展望

### 项目总结

#### 实现的功能
- 使用四个 HelloAgents `SimpleAgent` 完成职责隔离的顺序协作。
- 使用两个官方 `Tool` 完成需求前置检查和报告后置质检。
- 提供 CLI、Notebook、示例输入、示例输出和离线自动化测试。

#### 遇到的挑战
- 模糊需求容易诱发隐含假设：通过角色提示词强制区分事实、建议和待确认项。
- LLM 输出不稳定：通过固定报告标题和确定性结构质检提供护栏。
- 普通测试不应依赖密钥：编排层允许注入离线替身，Notebook 也支持无密钥执行。

#### 未来改进方向
- 支持用户回答澄清问题后的增量迭代。
- 增加 JSON Schema 输出和失败自动修复。
- 使用标注集评估事实与假设的分类质量。